## make very toy data

## notes:
- The pixel scale is anisotropic, apparently?
- What size are the 2-d images?
- Notation is w,z is the coordinate system in the data images.

In [ ]:
import numpy as np
import pylab as plt
from astropy.table import Table

In [ ]:
rng = np.random.default_rng(17) # the most random number
SQRT2PI = np.sqrt(2. * np.pi)

In [ ]:
def choose_xy(rng, rad=0.59):
    xy = np.inf, np.inf
    while np.linalg.norm(xy) > rad:
        xy = rad * (2. * rng.uniform(size=2) - 1.)
    return xy

def oned_gaussian(xs, mu, sigma=1.):
    return (1. / (SQRT2PI * sigma)) * np.exp(-0.5 * ((xs - mu) / sigma) ** 2)

def rotscale(psi):
    """
    ## input:
    - `psi`: a scan angle

    ## output:
    - the 2x2 operator that transforms an x,y displacement on the sky to a w,z displacement on the device

    ## comments:
    - The pixels aren't square, goddamn it.

    ## bugs:
    - Hard-coded magic pixel scales.
    """
    wpixscale = 0.059 # arcsec
    zpixscale = 3. * wpixscale 
    cp, sp = np.cos(psi), np.sin(psi)
    return np.array([[cp / wpixscale, sp / wpixscale], [-sp / zpixscale, cp / zpixscale]])

def make_one_toy_dataset(rng):
    """
    ## inputs:
    - `rng`: A numpy random number generator

    ## outputs:
    - `X`: A table of housekeeping data
    - `Y`: A set of Gaia-like images
    - `truth`: A table of true information about the scene
    
    ## bugs:
    - Everything is hard-coded.
    - Cadence, scan-angle distribution, CCD numbers, are all unrealistically uncorrelated, and thus this is too easy.
    """
    Nstars = 0
    while Nstars < 1:
        Nstars = rng.poisson(1.5) # magic
    # print(Nstars)
    truth = Table()
    truth["flux"] = np.sort(1. + 10. * rng.uniform(size=Nstars) ** 2)[::-1] # so much magic
    truth["xy"] = np.atleast_2d([np.zeros(2), ] + [choose_xy(rng) for i in range(Nstars - 1)])
    # print(truth)
    N, Nw, Nz = 120, 18, 12 # this seems wrong, but Hogg has read a lot of documentation
    X = Table()
    X["time"] = np.sort(rng.uniform(size=N)) * 5. * 365.25
    X["scan_angle"] = rng.uniform(size=N) * 2. * np.pi
    X["wz_offset"] = 1.5 * (2. * rng.uniform(size=(N, 2)) - 1.) # magic
    X["CCD"] = rng.choice(np.arange(9 * 6), size=N, replace=True) # wrong number of CCDs?
    # print(X)
    Y = 0.01 * rng.normal(size=(N, Nw, Nz)) # magic: noise variance
    ws, zs = np.arange(Nw) - 0.5 * (Nw - 1.), np.arange(Nz) - 0.5 * (Nz - 1.)
    for j in range(N):
        for q in range(Nstars):
            wz = rotscale(X["scan_angle"][j]) @ (truth["xy"][q]) + X["wz_offset"][j]
            Y[j] += truth["flux"][q] * oned_gaussian(ws, wz[0])[:, None] * oned_gaussian(zs, wz[1])[None, :]
    return X, Y, truth

In [ ]:
X, Y, truth = make_one_toy_dataset(rng)
print(truth)
print(Y.shape)

In [ ]:
N, Nw, Nz = Y.shape
w0, z0 = 0.5 * (Nw - 1.), 0.5 * (Nz - 1.)
foo = np.array([1., 0.]) # arcsec
for j in range(5):
    plt.imshow(Y[j].T, origin="lower", interpolation="nearest", cmap="gray",
               aspect=3.)
    dw, dz = X["wz_offset"][j]
    plt.scatter(w0 + dw, z0 + dz, marker="x", c="r", s=50, lw=2)
    bar = rotscale(X["scan_angle"][j]) @ foo
    plt.plot([w0 + dw, w0 + dw + bar[0]], [z0 + dz, z0 + dz + bar[1]], "r-", lw=0.5)
    plt.xlim(-0.5, Nw - 0.5)
    plt.ylim(-0.5, Nz - 0.5)
    plt.show()